# MuscleMap WB Segmentation — Water images (Lambda)

Runs MuscleMap whole-body segmentation on **Dixon WATER** stacks.
Model weights are bundled in the pip package — no separate upload needed.

MuscleMap requires Python 3.11. This notebook creates a conda env with Python 3.11
and installs MuscleMap there (Lambda's default Python 3.10 is not supported).

## Before running

Upload data:
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  your machine9:~/
```

## Download results when done
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine9:~/musclemap_water_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/musclemap_water_segs/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os, shutil

# Search common conda locations on Lambda/Ubuntu
_conda_candidates = [
    shutil.which('conda'),
    os.path.expanduser('~/miniconda3/bin/conda'),
    os.path.expanduser('~/anaconda3/bin/conda'),
    '/opt/conda/bin/conda',
    '/usr/local/conda/bin/conda',
]
CONDA = next((p for p in _conda_candidates if p and os.path.exists(p)), None)

if CONDA is None:
    print('conda not found — installing Miniconda ...')
    subprocess.check_call([
        'bash', '-c',
        'wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh '
        '-O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -p ~/miniconda3'
    ])
    CONDA = os.path.expanduser('~/miniconda3/bin/conda')

print('conda:', CONDA)

ENV_NAME     = 'musclemap_env'
conda_prefix = os.path.dirname(os.path.dirname(CONDA))
ENV_DIR      = os.path.join(conda_prefix, 'envs', ENV_NAME)
ENV_PY       = os.path.join(ENV_DIR, 'bin', 'python')

# Create Python 3.11 env using conda-forge (no Anaconda TOS required)
if not os.path.exists(ENV_PY):
    print('Creating conda env with Python 3.11 ...')
    subprocess.check_call([
        CONDA, 'create', '-n', ENV_NAME, 'python=3.11', 'pip',
        '-c', 'conda-forge', '--override-channels',
        '-y', '-q',
    ])
    print('Env created.')
else:
    print('Conda env already exists.')

# Use 'python -m pip' — works regardless of where the pip binary lives
def env_pip(*args):
    subprocess.check_call([ENV_PY, '-m', 'pip'] + list(args))

# Ensure pip is available
env_pip('install', '--upgrade', '-q', 'pip')

# Install MuscleMap
env_pip('install', '-q', 'git+https://github.com/MuscleMap/MuscleMap.git')

MM_BIN = os.path.join(ENV_DIR, 'bin', 'mm_segment')
print('mm_segment exists:', os.path.exists(MM_BIN))

In [ ]:
import glob
import os
import SimpleITK as sitk
import numpy as np

In [ ]:
import glob

# --- paths ---
DATA_ROOT  = os.path.expanduser('~/myosegmenTUM')
OUTPUT_DIR = os.path.expanduser('~/musclemap_water_segs')
IMAGE_GLOB = os.path.join(DATA_ROOT, '*', 'ImageData', '*_WATER', '*_WATER_stack*.nii')

os.makedirs(OUTPUT_DIR, exist_ok=True)

image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} WATER stacks')
for p in image_files:
    print(' ', p)

In [ ]:
# Clear the Jupyter-set MPLBACKEND so matplotlib works in the conda subprocess
run_env = os.environ.copy()
run_env.pop('MPLBACKEND', None)

# run MuscleMap WB on each stack using the Python 3.11 conda env
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]  # e.g. HV001_1_WATER_stack1
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dseg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (already done): {stem}')
        continue

    print(f'\nProcessing: {nii_path}')
    subprocess.check_call([
        MM_BIN,
        '-i', nii_path,
        '-r', 'wholebody',
        '-o', OUTPUT_DIR,
        '-g', 'Y',
    ], env=run_env)
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
# quick sanity check — reload one result and show labels present
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_dseg.nii.gz')))
print(f'Total output files: {len(results)}')
if results:
    sample_sitk = sitk.ReadImage(results[0])
    sample_arr  = sitk.GetArrayFromImage(sample_sitk)
    print(f'Sample : {results[0]}')
    print(f'Shape  : {sample_arr.shape}')
    labels = sorted(np.unique(sample_arr[sample_arr > 0]).tolist())
    print(f'Labels : {labels}')